# 02 - Baseline de cohérence image-légende

Cette baseline prépare la tâche de l'option B du projet : décider si une
légende correspond à une image. Elle crée des paires correctes et
incorrectes à partir des mêmes 500 images Flickr8k. Aucun VLM n'est utilisé
à cette étape.


In [1]:
# Installer uniquement la bibliothèque qui peut manquer dans Colab.
!pip -q install datasets

In [2]:
# Connecter Google Drive et créer les dossiers du projet.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ia_visuelle")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [3]:
from datasets import load_dataset
import pandas as pd

SEED = 42
N_IMAGES = 500

# Reprendre exactement la même sélection que dans le notebook 01.
dataset = load_dataset("jxie/flickr8k", split="train")
sample_size = min(N_IMAGES, len(dataset))
sample = dataset.shuffle(seed=SEED).select(range(sample_size))
print(f"Images utilisées : {len(sample)}")


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00002-2f8f6bfa852eac(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-2173151d8cd6c7(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00001-of-00002-2173151d8cd6c7(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-7025a2b59(…): reconstructing file:   0%|          |  0.00B /  138MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Images utilisées : 500


In [ ]:
# Construire des paires positives et négatives.
# Une paire positive associe une image à sa propre légende.
# Une paire négative associe l'image à la légende de l'image suivante.
pairs = []

for index, example in enumerate(sample):
    positive_caption = example["caption_0"]
    negative_index = (index + 1) % len(sample)
    negative_caption = sample[negative_index]["caption_0"]

    pairs.append({
        "image_id": index,
        "caption": positive_caption,
        "label": 1,
        "pair_type": "correct",
    })
    pairs.append({
        "image_id": index,
        "caption": negative_caption,
        "label": 0,
        "pair_type": "incorrect",
    })

pairs_df = pd.DataFrame(pairs)
pairs_df.head()

In [ ]:
# Une baseline al?atoire a une pr?cision attendue de 50 %.
# Elle sert de point de comparaison pour BLIP, GIT et les humains.
baseline_accuracy = 0.5

metrics = pd.DataFrame([{
    "experiment": "random_baseline",
    "accuracy": baseline_accuracy,
}])

pairs_path = RESULTS_DIR / "baseline_alignment_pairs.csv"
metrics_path = RESULTS_DIR / "baseline_metrics.csv"
pairs_df.to_csv(pairs_path, index=False)
metrics.to_csv(metrics_path, index=False)

print(f"Paires sauvegard?es dans : {pairs_path}")
print(f"M?triques sauvegard?es dans : {metrics_path}")
metrics

## Interprétation

Cette baseline ne juge pas les images elle-même. Elle fournit un point de
départ à dépasser. Les prochains notebooks demanderont à BLIP et GIT de
juger les mêmes paires image-légende, puis compareront leurs décisions aux
évaluations humaines.
